In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_PATH = "/content/drive/MyDrive/DiffSynth-Studio"
DRIVE_PYTHON_PATH = DRIVE_PATH.replace("\\","")

if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

SYM_PATH = "/content/DiffSynth-Studio"
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH




In [ ]:
# ! git clone https://github.com/modelscope/DiffSynth-Studio.git
%cd DiffSynth-Studio
! pip install -e .

In [ ]:
import os
os.environ["MODELSCOPE_DOMAIN"] = "www.modelscope.ai"

In [ ]:
import wandb
wandb.login()


# Build DataSet

In [ ]:
!cp -r /content/DiffSynth-Studio /content/drive/MyDrive

In [ ]:
!ls /content/drive/MyDrive/DPO

In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/DPO/scripts_0122_2026")

scripts = {}

for file in folder.rglob("*.txt"):
    with open(file, "r", encoding="utf-8") as f:
        scripts[file.stem] = f.read()

print(scripts)

In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/DPO/all_videos")

videos = {}

for file in folder.rglob("*.mp4"):
  if file.stem in scripts.keys():
    videos[file.stem] = str(file)

print(videos)

In [ ]:
pd.Da

In [ ]:
! ls /content/drive/MyDrive/DiffSynth-Studio/data

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/DiffSynth-Studio/data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/metadata.csv")

In [ ]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

In [ ]:
df.iloc[:3].to_csv("/content/DiffSynth-Studio/data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/metadata.csv")

In [ ]:
import pandas as pd
df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/metadata.csv")

train_df = df.sample(frac=0.8, random_state=42)
remaining = df.drop(train_df.index)
val_df = remaining.sample(frac=0.5, random_state=42)  # 10%
test_df = remaining.drop(val_df.index)  # 10%

train_df.to_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/train_metadata.csv")
val_df.to_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv")
test_df.to_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/test_metadata.csv")



# Fine Tune

In [ ]:
# @title 5B Full Image amd text

! accelerate launch examples/wanvideo/model_training/train.py \
  --dataset_base_path "" \
  --dataset_metadata_path data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/train_metadata.csv \
  --height 832 \
  --width 480 \
  --num_frames 121 \
  --dataset_repeat 1 \
  --model_id_with_origin_paths "Wan-AI/Wan2.2-TI2V-5B-BF16:diffusion_pytorch_model*.safetensors,Wan-AI/Wan2.2-TI2V-5B-BF16:models_t5_umt5-xxl-enc-bf16.pth,Wan-AI/Wan2.2-TI2V-5B-BF16:Wan2.2_VAE.pth" \
  --learning_rate 1e-4 \
  --num_epochs 2 \
  --remove_prefix_in_ckpt "pipe.dit." \
  --output_path "./models/train/Wan2.2-TI2V-5B_full_LR1e04" \
  --trainable_models "dit" \
  --extra_inputs "input_image"


In [ ]:
# @title 5B Image
!accelerate launch examples/wanvideo/model_training/train.py \
  --dataset_base_path "" \
  --dataset_metadata_path data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/train_metadata.csv \
  --height 832 \
  --width 480 \
  --num_frames 121 \
  --dataset_repeat 1 \
  --model_id_with_origin_paths "Wan-AI/Wan2.2-TI2V-5B-BF16:diffusion_pytorch_model*.safetensors,Wan-AI/Wan2.2-TI2V-5B-BF16:models_t5_umt5-xxl-enc-bf16.pth,Wan-AI/Wan2.2-TI2V-5B-BF16:Wan2.2_VAE.pth" \
  --learning_rate 1e-4 \
  --num_epochs 1 \
  --remove_prefix_in_ckpt "pipe.dit." \
  --output_path "./models/train/Wan2.2-TI2V-5B_lora" \
  --lora_base_model "dit" \
  --lora_target_modules "q,k,v,o,ffn.0,ffn.2" \
  --lora_rank 64 \
  --extra_inputs "input_image" \
  --task "sft"

In [ ]:
# @title 5B Text
!accelerate launch examples/wanvideo/model_training/train.py \
  --dataset_base_path "" \
  --dataset_metadata_path data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/train_metadata.csv \
  --height 832 \
  --width 480 \
  --num_frames 121 \
  --dataset_repeat 1 \
  --model_id_with_origin_paths "Wan-AI/Wan2.2-TI2V-5B-BF16:diffusion_pytorch_model*.safetensors,Wan-AI/Wan2.2-TI2V-5B-BF16:models_t5_umt5-xxl-enc-bf16.pth,Wan-AI/Wan2.2-TI2V-5B-BF16:Wan2.2_VAE.pth" \
  --learning_rate 1e-4 \
  --num_epochs 1 \
  --remove_prefix_in_ckpt "pipe.dit." \
  --output_path "./models/train/Wan2.2-TI2V-5B_lora_wo_image" \
  --lora_base_model "dit" \
  --lora_target_modules "q,k,v,o,ffn.0,ffn.2" \
  --lora_rank 64 \
  --task "sft"

In [ ]:
# @title A14B
!modelscope download --dataset DiffSynth-Studio/diffsynth_example_dataset --include "wanvideo/Wan2.2-I2V-A14B/*" --local_dir ./data/diffsynth_example_dataset

!accelerate launch examples/wanvideo/model_training/train.py \
  --dataset_base_path data/diffsynth_example_dataset/wanvideo/Wan2.2-I2V-A14B \
  --dataset_metadata_path data/diffsynth_example_dataset/wanvideo/Wan2.2-I2V-A14B/metadata.csv \
  --height 832 \
  --width 480 \
  --num_frames 49 \
  --dataset_repeat 100 \
  --model_id_with_origin_paths "Wan-AI/Wan2.2-I2V-A14B:high_noise_model/diffusion_pytorch_model*.safetensors,Wan-AI/Wan2.2-I2V-A14B:models_t5_umt5-xxl-enc-bf16.pth,Wan-AI/Wan2.2-I2V-A14B:Wan2.1_VAE.pth" \
  --learning_rate 1e-4 \
  --num_epochs 5 \
  --remove_prefix_in_ckpt "pipe.dit." \
  --output_path "./models/train/Wan2.2-I2V-A14B_high_noise_lora" \
  --lora_base_model "dit" \
  --lora_target_modules "q,k,v,o,ffn.0,ffn.2" \
  --lora_rank 32 \
  --extra_inputs "input_image" \
  --max_timestep_boundary 0.358 \
  --min_timestep_boundary 0
# boundary corresponds to timesteps [900, 1000]

!accelerate launch examples/wanvideo/model_training/train.py \
  --dataset_base_path data/diffsynth_example_dataset/wanvideo/Wan2.2-I2V-A14B \
  --dataset_metadata_path data/diffsynth_example_dataset/wanvideo/Wan2.2-I2V-A14B/metadata.csv \
  --height 480 \
  --width 832 \
  --num_frames 49 \
  --dataset_repeat 100 \
  --model_id_with_origin_paths "Wan-AI/Wan2.2-I2V-A14B:low_noise_model/diffusion_pytorch_model*.safetensors,Wan-AI/Wan2.2-I2V-A14B:models_t5_umt5-xxl-enc-bf16.pth,Wan-AI/Wan2.2-I2V-A14B:Wan2.1_VAE.pth" \
  --learning_rate 1e-4 \
  --num_epochs 5 \
  --remove_prefix_in_ckpt "pipe.dit." \
  --output_path "./models/train/Wan2.2-I2V-A14B_low_noise_lora" \
  --lora_base_model "dit" \
  --lora_target_modules "q,k,v,o,ffn.0,ffn.2" \
  --lora_rank 32 \
  --extra_inputs "input_image" \
  --max_timestep_boundary 1 \
  --min_timestep_boundary 0.358
# boundary corresponds to timesteps [0, 900)


# Inference

In [ ]:
import torch
from PIL import Image
from diffsynth.pipelines.wan_video import WanVideoPipeline, ModelConfig
from modelscope import dataset_snapshot_download
from diffsynth.utils.data import save_video, VideoData
from diffsynth.core import load_state_dict

pipe = WanVideoPipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda",
    model_configs=[
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B-BF16", origin_file_pattern="models_t5_umt5-xxl-enc-bf16.pth"),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B-BF16", origin_file_pattern="diffusion_pytorch_model*.safetensors"),
        ModelConfig(model_id="Wan-AI/Wan2.2-TI2V-5B-BF16", origin_file_pattern="Wan2.2_VAE.pth"),
    ],
    tokenizer_config=ModelConfig(model_id="Wan-AI/Wan2.1-T2V-1.3B", origin_file_pattern="google/umt5-xxl/"),
)
# pipe.load_lora(
#     pipe.dit,   # 👈 关键：LoRA 加在 DiT 上
#     "/content/DiffSynth-Studio/models/train/Wan2.2-TI2V-5B_lora_wo_text/epoch-0.safetensors",
#     alpha=1.0
# )


# state_dict = load_state_dict("/content/DiffSynth-Studio/models/train/Wan2.2-TI2V-5B_full/epoch-1.safetensors")
# pipe.dit.load_state_dict(state_dict)

In [ ]:
from PIL import Image
import cv2

def get_first_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

In [ ]:
from os.path import exists
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

save_dir = "val_wo_image"
os.makedirs(save_dir, exist_ok=True)

for idx, row in val_df.iterrows():
  video_path = row["video"]
  prompt = row["prompt"]
  id = row["id"]
  negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
  input_image = get_first_frame(video_path).resize((480, 832))
  video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=0, tiled=True,
    height=832, width=480,
    # input_image=input_image,
    num_frames=121,
  )
  save_video(video,f"{save_dir}/{id}.mp4", fps=15, quality=5)



In [ ]:
from os.path import exists
import pandas as pd
from tqdm.auto import tqdm
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

save_dir = "val_wo_text"
os.makedirs(save_dir, exist_ok=True)

for idx, row in tqdm(val_df.iterrows()):
  video_path = row["video"]
  prompt = " "
  id = row["id"]
  negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
  input_image = get_first_frame(video_path).resize((480, 832))
  video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=0, tiled=True,
    height=832, width=480,
    input_image=input_image,
    num_frames=121,
  )
  save_video(video,f"{save_dir}/{id}.mp4", fps=15, quality=5)


In [ ]:
from os.path import exists
import pandas as pd
from tqdm.auto import tqdm
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

save_dir = "val_wo_train"
os.makedirs(save_dir, exist_ok=True)

for idx, row in tqdm(val_df.iterrows()):
  video_path = row["video"]
  prompt = row["prompt"]
  id = row["id"]
  negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
  input_image = get_first_frame(video_path).resize((480, 832))
  video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=0, tiled=True,
    height=832, width=480,
    # input_image=input_image,
    num_frames=121,
  )
  save_video(video,f"{save_dir}/{id}.mp4", fps=15, quality=5)



In [ ]:
pipe.load_lora(
    pipe.dit,   # 👈 关键：LoRA 加在 DiT 上
    "/content/DiffSynth-Studio/models/train/Wan2.2-TI2V-5B_lora/epoch-0.safetensors",
    alpha=1.0
)

In [ ]:
from os.path import exists
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

save_dir = "val_with_it"
os.makedirs(save_dir, exist_ok=True)

for idx, row in tqdm(val_df.iterrows()):
  video_path = row["video"]
  prompt = row["prompt"]
  id = row["id"]
  negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
  input_image = get_first_frame(video_path).resize((480, 832))
  video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=0, tiled=True,
    height=832, width=480,
    input_image=input_image,
    num_frames=121,
  )
  save_video(video,f"{save_dir}/{id}.mp4", fps=15, quality=5)



In [ ]:
from os.path import exists
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

save_dir = "val_with_it_full"
os.makedirs(save_dir, exist_ok=True)

for idx, row in val_df.iterrows():
  video_path = row["video"]
  prompt = row["prompt"]
  id = row["id"]
  negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
  input_image = get_first_frame(video_path).resize((480, 832))
  video = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    seed=0, tiled=True,
    height=832, width=480,
    input_image=input_image,
    num_frames=121,
  )
  save_video(video,f"{save_dir}/{id}.mp4", fps=15, quality=5)



In [ ]:
# @title FVD

"""
Self-contained FVD-style score between two sets of .mp4 videos.

No external FVD packages — uses torchvision's r3d_18 (Kinetics-400 pretrained)
as the feature extractor. This is not bit-exact to the original I3D-based FVD,
but it gives a valid Fréchet distance in a video-feature space and is
comparable across your own experiments (e.g., Wan2.2 ckpt A vs ckpt B).

Deps:
    pip install torch torchvision decord scipy numpy

If you later want canonical FVD numbers for a paper, swap in an I3D checkpoint
at the feature extractor — the rest of the pipeline is unchanged.
"""

import numpy as np
import torch
import torch.nn as nn
from scipy import linalg
from decord import VideoReader, cpu
from torchvision.models.video import r3d_18, R3D_18_Weights
from typing import List, Tuple


# ---- Kinetics normalization stats (same as torchvision R3D_18 transforms) ----
_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(1, 3, 1, 1, 1)
_STD  = torch.tensor([0.22803, 0.22145, 0.216989]).view(1, 3, 1, 1, 1)


def _load_clip(path: str, num_frames: int, size: int) -> torch.Tensor:
    """Return (C, T, H, W) float tensor in [0, 1], uniformly sampled."""
    vr = VideoReader(path, ctx=cpu(0), width=size, height=size)
    total = len(vr)
    if total >= num_frames:
        idx = np.linspace(0, total - 1, num_frames).astype(int)
    else:
        idx = np.concatenate([np.arange(total),
                              np.full(num_frames - total, total - 1)])
    frames = vr.get_batch(idx).asnumpy()           # (T, H, W, C) uint8
    t = torch.from_numpy(frames).float() / 255.0
    return t.permute(3, 0, 1, 2)                   # (C, T, H, W)


def _build_model(device: torch.device) -> nn.Module:
    m = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)
    m.fc = nn.Identity()                           # 512-d features
    return m.eval().to(device)


@torch.no_grad()
def _extract_features(paths: List[str],
                      model: nn.Module,
                      device: torch.device,
                      num_frames: int,
                      size: int,
                      batch_size: int) -> np.ndarray:
    mean, std = _MEAN.to(device), _STD.to(device)
    feats = []
    for i in range(0, len(paths), batch_size):
        clips = [_load_clip(p, num_frames, size) for p in paths[i:i + batch_size]]
        x = torch.stack(clips).to(device)          # (B, C, T, H, W)
        x = (x - mean) / std
        feats.append(model(x).cpu().numpy())
    return np.concatenate(feats, axis=0)


def _frechet(f1: np.ndarray, f2: np.ndarray, eps: float = 1e-6) -> float:
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1 = np.cov(f1, rowvar=False)
    s2 = np.cov(f2, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(s1 @ s2, disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(s1.shape[0]) * eps
        covmean = linalg.sqrtm((s1 + offset) @ (s2 + offset))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))


def compute_fvd(paths_a: List[str],
                paths_b: List[str],
                num_frames: int = 16,
                size: int = 112,
                batch_size: int = 8,
                n_bootstrap: int = 20,
                seed: int = 0,
                device: str = None) -> Tuple[float, float]:
    """
    Compute FVD-style score between two sets of videos.

    Args:
        paths_a, paths_b: lists of .mp4 paths
        num_frames: frames per clip (16 standard for r3d_18)
        size: spatial resolution (112 for r3d_18)
        batch_size: videos per forward pass
        n_bootstrap: bootstrap resamples for variance
        seed: RNG seed
        device: 'cuda' / 'cpu'; auto if None

    Returns:
        (mean, variance) over bootstrap samples.
        The first sample is the point estimate on the full sets.
    """
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model = _build_model(device)

    feat_a = _extract_features(paths_a, model, device, num_frames, size, batch_size)
    feat_b = _extract_features(paths_b, model, device, num_frames, size, batch_size)

    rng = np.random.default_rng(seed)
    scores = [_frechet(feat_a, feat_b)]           # point estimate
    n_a, n_b = len(feat_a), len(feat_b)
    for _ in range(n_bootstrap):
        ia = rng.integers(0, n_a, n_a)
        ib = rng.integers(0, n_b, n_b)
        scores.append(_frechet(feat_a[ia], feat_b[ib]))

    scores = np.array(scores)
    return float(scores.mean()), float(scores.var(ddof=1))




In [ ]:
! pip install decord

In [ ]:
# @title Per-frame Cosine Similarity
"""
Per-frame cosine similarity between paired sets of videos.

Same feature pipeline as the KID script:
    - 121 frames @ 15 fps from generated videos.
    - 121 uniformly sampled frames from real videos (duration-stretched).
    - InceptionV3 pool3 (2048-d) features.

Two complementary metrics, both returned:

    1. Paired cosine (primary):
        For each pair i and frame t, compute
            cos(feats_real[i, t], feats_gen[i, t]).
        Aggregate across N pairs -> mean ± std at each t.
        Semantics: "How similar is each generated video to its OWN reference
        at frame t?". Sensitive to per-pair correspondence.

    2. Mean-feature cosine (companion):
        At each t, take the mean feature across the N real videos and the
        mean across the N generated videos, then compute one cosine between
        the two means.
        Semantics: "How similar is the AVERAGE generated feature to the
        AVERAGE real feature at frame t?". Distribution-level, closer in
        spirit to FID/KID.

Both are bounded in [-1, 1]; for ReLU'd InceptionV3 features they sit in
[0, 1] in practice, with higher = better.
"""

from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from decord import VideoReader, cpu
from torchvision.models import inception_v3, Inception_V3_Weights


# ---- Constants (match the KID script) ----
NUM_FRAMES = 121
GEN_FPS = 15
INCEPTION_SIZE = 299

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


# -----------------------------------------------------------------------------
# Frame loading (identical to the KID script)
# -----------------------------------------------------------------------------

def _sample_frames_gen(path: str, num_frames: int, fps: int) -> np.ndarray:
    vr = VideoReader(path, ctx=cpu(0))
    total = len(vr)
    src_fps = vr.get_avg_fps() or fps
    times = np.arange(num_frames) / fps
    idx = np.round(times * src_fps).astype(int)
    idx = np.clip(idx, 0, total - 1)
    return vr.get_batch(idx).asnumpy()


def _sample_frames_real(path: str, num_frames: int) -> np.ndarray:
    vr = VideoReader(path, ctx=cpu(0))
    total = len(vr)
    if total >= num_frames:
        idx = np.linspace(0, total - 1, num_frames).astype(int)
    else:
        idx = np.concatenate([
            np.arange(total),
            np.full(num_frames - total, total - 1),
        ])
    return vr.get_batch(idx).asnumpy()


def _to_inception_input(frames: np.ndarray, device: torch.device) -> torch.Tensor:
    t = torch.from_numpy(frames).float().to(device) / 255.0
    t = t.permute(0, 3, 1, 2)
    t = F.interpolate(t, size=INCEPTION_SIZE, mode='bilinear', align_corners=False)
    t = (t - _MEAN.to(device)) / _STD.to(device)
    return t


# -----------------------------------------------------------------------------
# Feature extractor
# -----------------------------------------------------------------------------

def _build_inception(device: torch.device) -> nn.Module:
    model = inception_v3(
        weights=Inception_V3_Weights.IMAGENET1K_V1,
        aux_logits=True,
    )
    model.fc = nn.Identity()
    model.aux_logits = False
    return model.eval().to(device)


@torch.no_grad()
def _extract_video_features(
    path: str,
    is_generated: bool,
    model: nn.Module,
    device: torch.device,
    num_frames: int,
    fps: int,
    frame_batch: int,
) -> np.ndarray:
    if is_generated:
        frames = _sample_frames_gen(path, num_frames, fps)
    else:
        frames = _sample_frames_real(path, num_frames)
    x = _to_inception_input(frames, device)
    feats = []
    for i in range(0, x.shape[0], frame_batch):
        feats.append(model(x[i:i + frame_batch]).cpu().numpy())
    return np.concatenate(feats, axis=0)


# -----------------------------------------------------------------------------
# Cosine similarity helpers
# -----------------------------------------------------------------------------

def _row_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-normalize along the last axis."""
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, eps)


def _paired_cosine_per_frame(
    feats_real: np.ndarray, feats_gen: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    feats_real, feats_gen: (N, T, D).
    Returns:
        mean_t: (T,) mean cosine across N pairs at each frame.
        std_t:  (T,) std across N pairs at each frame.
        per_pair: (N, T) cosine for every (pair, frame) cell.
    """
    r = _row_normalize(feats_real)            # (N, T, D)
    g = _row_normalize(feats_gen)             # (N, T, D)
    per_pair = (r * g).sum(axis=-1)           # (N, T)
    mean_t = per_pair.mean(axis=0)            # (T,)
    std_t = per_pair.std(axis=0, ddof=1) if per_pair.shape[0] > 1 \
        else np.zeros_like(mean_t)
    return mean_t, std_t, per_pair


def _mean_feature_cosine_per_frame(
    feats_real: np.ndarray, feats_gen: np.ndarray,
) -> np.ndarray:
    """
    feats_real, feats_gen: (N, T, D).
    Returns: (T,) cosine between mean-over-N real feature and mean-over-N
    generated feature at each frame.
    """
    mu_r = feats_real.mean(axis=0)                          # (T, D)
    mu_g = feats_gen.mean(axis=0)                           # (T, D)
    mu_r_n = _row_normalize(mu_r)
    mu_g_n = _row_normalize(mu_g)
    return (mu_r_n * mu_g_n).sum(axis=-1)                   # (T,)


# -----------------------------------------------------------------------------
# Public API
# -----------------------------------------------------------------------------

def compute_per_frame_cosine(
    paths_real: List[str],
    paths_gen: List[str],
    num_frames: int = NUM_FRAMES,
    fps: int = GEN_FPS,
    frame_batch: int = 32,
    device: Optional[str] = None,
    verbose: bool = True,
) -> dict:
    """
    Per-frame cosine similarity between paired real and generated videos.

    Returns a dict with:
        'paired_mean':   (T,) np.ndarray, mean paired cosine per frame.
        'paired_std':    (T,) np.ndarray, std across pairs per frame.
        'paired_per_pair': (N, T) np.ndarray, raw per-pair values.
        'mean_feature':  (T,) np.ndarray, cosine between mean features.
        'feats':         (2, N, T, 2048) np.ndarray, feats[0]=real, feats[1]=gen.
    """
    assert len(paths_real) == len(paths_gen), \
        f"paired inputs must match: {len(paths_real)} vs {len(paths_gen)}"
    n_pairs = len(paths_real)
    if n_pairs < 1:
        raise ValueError("Need at least 1 pair.")

    device = torch.device(
        device or ('cuda' if torch.cuda.is_available() else 'cpu')
    )
    model = _build_inception(device)

    feats_real = np.zeros((n_pairs, num_frames, 2048), dtype=np.float32)
    feats_gen = np.zeros((n_pairs, num_frames, 2048), dtype=np.float32)

    for i, (pr, pg) in enumerate(zip(paths_real, paths_gen)):
        feats_real[i] = _extract_video_features(
            pr, is_generated=False, model=model, device=device,
            num_frames=num_frames, fps=fps, frame_batch=frame_batch,
        )
        feats_gen[i] = _extract_video_features(
            pg, is_generated=True, model=model, device=device,
            num_frames=num_frames, fps=fps, frame_batch=frame_batch,
        )
        if verbose:
            print(f"[{i + 1}/{n_pairs}] features extracted")

    paired_mean, paired_std, paired_per_pair = _paired_cosine_per_frame(
        feats_real, feats_gen,
    )
    mean_feature = _mean_feature_cosine_per_frame(feats_real, feats_gen)

    feats = np.stack([feats_real, feats_gen], axis=0)
    return {
        'paired_mean': paired_mean,
        'paired_std': paired_std,
        'paired_per_pair': paired_per_pair,
        'mean_feature': mean_feature,
        'feats': feats,
    }


# -----------------------------------------------------------------------------
# If you already have features cached (e.g. from the KID run), skip extraction
# -----------------------------------------------------------------------------

@torch.no_grad()
def compute_per_frame_cosine(
    paths_real: List[str],
    paths_gen: List[str],
    num_frames: int = NUM_FRAMES,
    fps: int = GEN_FPS,
    frame_batch: int = 32,
    device: Optional[str] = None,
    verbose: bool = True,
) -> List[float]:
    """
    Per-frame cosine similarity, averaged across paired videos.

    Returns:
        list of length num_frames, mean cosine at each frame index t.
    """
    assert len(paths_real) == len(paths_gen)
    n_pairs = len(paths_real)
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    model = _build_inception(device)

    feats_real = np.zeros((n_pairs, num_frames, 2048), dtype=np.float32)
    feats_gen = np.zeros((n_pairs, num_frames, 2048), dtype=np.float32)

    for i, (pr, pg) in enumerate(zip(paths_real, paths_gen)):
        feats_real[i] = _extract_video_features(
            pr, is_generated=False, model=model, device=device,
            num_frames=num_frames, fps=fps, frame_batch=frame_batch,
        )
        feats_gen[i] = _extract_video_features(
            pg, is_generated=True, model=model, device=device,
            num_frames=num_frames, fps=fps, frame_batch=frame_batch,
        )
        if verbose:
            print(f"[{i + 1}/{n_pairs}] features extracted")

    r = feats_real / np.maximum(np.linalg.norm(feats_real, axis=-1, keepdims=True), 1e-12)
    g = feats_gen / np.maximum(np.linalg.norm(feats_gen, axis=-1, keepdims=True), 1e-12)
    cos_per_frame = (r * g).sum(axis=-1).mean(axis=0)  # (T,)

    return cos_per_frame.tolist()

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

real = val_df["video"].to_list()
folder = "val_wo_image"
gen = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]
cs_1 = compute_per_frame_cosine(real, gen)

folder = "val_wo_text"
gen = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]
cs_2 = compute_per_frame_cosine(real, gen)


folder = "val_wo_train"
gen = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]
cs_3 = compute_per_frame_cosine(real, gen)


folder = "val_with_it"
gen = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]
cs_4 = compute_per_frame_cosine(real, gen)

folder = "val_with_it_full"
gen = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]
cs_5 = compute_per_frame_cosine(real, gen)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(cs_1, marker='o', label='w/o image')
plt.plot(cs_2, marker='s', label='w/o text')
plt.plot(cs_3, marker='^', label='w/o train')
plt.plot(cs_4, marker='d', label='with image & text')
plt.plot(cs_5, marker='d', label='with image & text & Full')

plt.xlabel('Frame Index')
plt.ylabel('KID')
plt.title('Per-frame Cosine Simlarity across Ablation Conditions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('per_frame_kid.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

ground_truth_paths = val_df["video"].to_list()
folder = "val_with_it_full"
paths_b = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]

mean_fvd, var_fvd = compute_fvd(
    ground_truth_paths, paths_b
)
print(f"FVD mean: {mean_fvd:.4f}, variance: {var_fvd:.4f}")

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

ground_truth_paths = val_df["video"].to_list()
folder = "val_wo_image"
paths_b = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]

mean_fvd, var_fvd = compute_fvd(
    ground_truth_paths, paths_b
)
print(f"FVD mean: {mean_fvd:.4f}, variance: {var_fvd:.4f}")

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

ground_truth_paths = val_df["video"].to_list()
folder = "val_wo_text"
paths_b = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]

mean_fvd, var_fvd = compute_fvd(
    ground_truth_paths, paths_b
)
print(f"FVD mean: {mean_fvd:.4f}, variance: {var_fvd:.4f}")

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

ground_truth_paths = val_df["video"].to_list()
folder = "val_wo_train"
paths_b = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]

mean_fvd, var_fvd = compute_fvd(
    ground_truth_paths, paths_b
)
print(f"FVD mean: {mean_fvd:.4f}, variance: {var_fvd:.4f}")

In [ ]:
import os
import pandas as pd
val_df = pd.read_csv("data/short_video_dataset/wanvideo/Wan2.2-TI2V-5B/val_metadata.csv").iloc[:10]

ground_truth_paths = val_df["video"].to_list()
folder = "val_with_it"
paths_b = [f"/content/DiffSynth-Studio/{folder}/{id}.mp4" for id in val_df["id"]]

mean_fvd, var_fvd = compute_fvd(
    ground_truth_paths, paths_b
)
print(f"FVD mean: {mean_fvd:.4f}, variance: {var_fvd:.4f}")